In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, floor, months_between, get
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.dim_customer_address")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "dim_customer_address"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_customer_address.ipynb"
_silver_table = "silver.customer_address"
_bad_table = f"bad_{_table_name}"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.dim_customer_address


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 4
+-----------+------+-------+----------+------------+----------------+-------------+-------------+-----+-------+-------+--------------------+----------+----------+----------------+--------------------------+-----------------------------+--------------------------+-----------------------------+
|customer_id|store |channel|address_id|address_type|address_line1   |address_line2|city         |state|zip    |country|effective_start_date|is_current|is_deleted|source_file     |created_on                |created_by                   |modified_on               |modified_by                  |
+-----------+------+-------+----------+------------+----------------+-------------+-------------+-----+-------+-------+--------------------+----------+----------+----------------+--------------------------+-----------------------------+--------------------------+-----------------------------+
|CUST001    |AMZ_US|MKT    |ADDR000001|BILLING     |123 Maple Street|Apt 4B    

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 4


In [10]:
# Generating surrogate key and business key

df_sk = df.withColumn("address_sk", xxhash64(concat_ws("||", "address_id", "customer_id", "store", "channel", "effective_start_date")).cast("bigint")) \
          .withColumn("address_bk", xxhash64(concat_ws("||", "address_id", "customer_id", "store", "channel")).cast("bigint")) \
          .withColumn("customer_bk", xxhash64(concat_ws("||", "customer_id", "store", "channel")).cast("bigint"))
    

df_sk.show(truncate = False)

+-----------+------+-------+----------+------------+----------------+-------------+-------------+-----+-------+-------+--------------------+----------+----------+----------------+--------------------------+-----------------------------+--------------------------+-----------------------------+--------------------+--------------------+--------------------+
|customer_id|store |channel|address_id|address_type|address_line1   |address_line2|city         |state|zip    |country|effective_start_date|is_current|is_deleted|source_file     |created_on                |created_by                   |modified_on               |modified_by                  |address_sk          |address_bk          |customer_bk         |
+-----------+------+-------+----------+------------+----------------+-------------+-------------+-----+-------+-------+--------------------+----------+----------+----------------+--------------------------+-----------------------------+--------------------------+-----------------------

In [11]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_gold_customer = spark.read.table("gold.dim_customer").where("is_current = true").select("customer_sk", "customer_bk")

df_invalid = df_sk.join(df_gold_customer, how = 'left_anti', on=df_sk.customer_bk==df_gold_customer.customer_bk)

print(f"SPARK-APP: Bad Record Count : {df_invalid.count()}")  

df_invalid.show(truncate = False)

if df_invalid.count() > 0:
    df_invalid.withColumn('error_description', lit("Customer not found in gold.dim_customer")) \
                       .select("address_id", "customer_id", "store", "channel", "address_type", "address_line1", "address_line2",
                               "city", "state", "zip", "country", "effective_start_date", "is_current", "is_deleted","created_on", 
                               "created_by", "modified_on","modified_by", "error_description") \
                       .write \
                       .format("delta") \
                       .mode("append") \
                       .saveAsTable(f"bad.{_bad_table}")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_customer, how = "inner", on=df_sk.customer_bk==df_gold_customer.customer_bk) \
                 .select("address_sk", "address_bk", "customer_sk", "address_id", "store_sk", "channel_sk", "address_type", "address_line1", "address_line2",
                         "city", "state", expr("zip as postal_code"), "country", "effective_start_date", "is_current", "is_deleted", "source_file")
                

print(f"SPARK-APP: Total records to be written to {_schema_name}.{_table_name} : {df_silver.count()}")

df_silver.show(truncate = False)

SPARK-APP: Bad Record Count : 1
+-----------+-----+-------+----------+------------+-------------+-------------+---------+-----+-------+-------+--------------------+----------+----------+----------------+--------------------------+-----------------------------+--------------------------+-----------------------------+-------------------+--------------------+--------------------+
|customer_id|store|channel|address_id|address_type|address_line1|address_line2|city     |state|zip    |country|effective_start_date|is_current|is_deleted|source_file     |created_on                |created_by                   |modified_on               |modified_by                  |address_sk         |address_bk          |customer_bk         |
+-----------+-----+-------+----------+------------+-------------+-------------+---------+-----+-------+-------+--------------------+----------+----------+----------------+--------------------------+-----------------------------+--------------------------+-----------------

In [12]:
#Display bad records

spark.read.table(f"bad.{_bad_table}").show()

+----------+-----------+-----+-------+------------+-------------+-------------+---------+-----+-------+-------+--------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|address_id|customer_id|store|channel|address_type|address_line1|address_line2|     city|state|    zip|country|effective_start_date|is_current|is_deleted|          created_on|          created_by|         modified_on|         modified_by|source_file|   error_description|
+----------+-----------+-----+-------+------------+-------------+-------------+---------+-----+-------+-------+--------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+-----------+--------------------+
|ADDR000004|    CUST004| EBAY|    MKT|     MAILING|  1801 street|         null|Bangalore|   KA|5600001|     IN|          2022-01-01|      true|     false|2026-01-29 00:53:...|silver_cu

In [13]:
# Truncate table for full load
dt_dim_customer_add = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_dim_customer_add.delete()
    dt_dim_customer_add.vacuum(0)



In [14]:
#SCD2 - Merge 1 for dates already in dim table
#Assuming file can have mixed cases for sending records - could be with history versions or sometimes only latest rows

dt_dim_customer_add.alias("t").merge(
    df_silver.alias("s"), 
    "t.address_bk = s.address_bk and t.is_current = true and s.effective_start_date > t.effective_start_date"
    ).whenMatchedUpdate(set={
    "effective_end_date" : "date_sub(s.effective_start_date, 1)",
    "is_current" : "false",
    "modified_on" : "current_timestamp()",
    "modified_by" : f"'{_script_name}'"
}).execute()

print("SPARK-APP: Merge1 completed")

SPARK-APP: Merge1 completed


In [15]:
dt_dim_customer_add.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows").show(1, False)

+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|0      |null           |null                 |null                |null         |
+-------+---------------+---------------------+--------------------+-------------+



In [16]:

window = Window.partitionBy("address_bk").orderBy("effective_start_date")

df_silver_nextDate = df_silver.withColumn("next_date", lead("effective_start_date").over(window)) \
                              .withColumn("effective_end_date", when(col("next_date").isNotNull(), date_sub("next_date",1)).otherwise(lit(None).cast("date"))) \
                              .withColumn("is_current", col("next_date").isNull()) \
                              .drop("next_date")

In [17]:
df_silver_nextDate.show()
df_silver_nextDate.printSchema()

+--------------------+--------------------+--------------------+----------+--------+----------+------------+----------------+-------------+-------------+-----+-----------+-------+--------------------+----------+----------+----------------+------------------+
|          address_sk|          address_bk|         customer_sk|address_id|store_sk|channel_sk|address_type|   address_line1|address_line2|         city|state|postal_code|country|effective_start_date|is_current|is_deleted|     source_file|effective_end_date|
+--------------------+--------------------+--------------------+----------+--------+----------+------------+----------------+-------------+-------------+-----+-----------+-------+--------------------+----------+----------+----------------+------------------+
| 8498463528029598722|-1821679000689094714| -798410774282725028|ADDR000001|       2|         1|     BILLING|123 Maple Street|       Apt 4B|      Seattle|   WA|      98101|     US|          2023-01-01|      true|     false|d

In [18]:
# Merge 2 for new dates not in dim table
dt_dim_customer_add.alias("t").merge(
    df_silver_nextDate.alias("s"),
    "t.address_bk = s.address_bk and s.effective_start_date = t.effective_start_date"
).whenNotMatchedInsert(values = {
    "address_sk": "s.address_sk", 
    "address_bk": "s.address_bk",
    "address_id": "s.address_id",
    "customer_sk": "s.customer_sk",
    "store_sk"   : "s.store_sk",
    "channel_sk" : "s.channel_sk",
    "address_type" : "s.address_type",
    "address_line1": "s.address_line1",
    "address_line2": "s.address_line2",
    "city"      : "s.city",
    "state"     : "s.state",
    "postal_code"   : "s.postal_code",
    "country" : "s.country",
    "effective_start_date" : "s.effective_start_date",
    "effective_end_date" : "s.effective_end_date",
    "is_current"  : "s.is_current",
    "is_deleted"  : "s.is_deleted",
    "source_file" : "s.source_file",
    "created_on"  : "current_timestamp()",
    "created_by"  : f"'{_script_name}'",
    "modified_on" : "current_timestamp()",
    "modified_by" : f"'{_script_name}'"
}).execute()
                     
print("SPARK-APP: Merge2 completed")    

SPARK-APP: Merge2 completed


In [19]:
hist = dt_dim_customer_add.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           4641|                    3|                   0|            3|
+-------+---------------+---------------------+--------------------+-------------+



In [20]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [21]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|layer|schema_name|          table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| gold|       gold|dim_customer_address|delta-load|     merge|2026-01-29 02:41:...|    success|           3|2026-01-29 02:41:...|gold_customer_add...|2026-01-29 02:41:...|gold_customer_add...|
+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [22]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------------+--------------------+----------+--------------------+--------+----------+------------+----------------+-------------+-------------+-----+-----------+-------+--------------------+------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+----------------+
|          address_sk|          address_bk|address_id|         customer_sk|store_sk|channel_sk|address_type|   address_line1|address_line2|         city|state|postal_code|country|effective_start_date|effective_end_date|is_current|is_deleted|          created_on|          created_by|         modified_on|         modified_by|     source_file|
+--------------------+--------------------+----------+--------------------+--------+----------+------------+----------------+-------------+-------------+-----+-----------+-------+--------------------+------------------+----------+----------+--------------------+--------------------+--------------------+----------

In [23]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [24]:
spark.stop()